In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from ira.ingest.ingest_scorecard import save, collect
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL

try:
    root = Path(__file__).resolve().parent
    os.chdir(root)
except NameError:
    root = Path.cwd()/"institutional-roi-analysis"
    os.chdir(root)

pd.set_option("display.max_columns",None)
display(Path.cwd())

WindowsPath('C:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

# Removed features
---
### 1

```
"latest.student.demographics.avg_family_income"
"latest.student.demographics.median_hh_income"
```
overlaps with ```"latest.student.demographics.median_family_income"```

---
### 2

```
"latest.academics.program_reporter.programs_offered"
```
~83% is null

---
### 3

```
latest.admissions.test_requirements
```
~46% is null and overlaps with ```latest.admissions.admission_rate.overall  ```

---
### 4

```
"latest.admissions.sat_scores.50th_percentile.critical_reading",
"latest.admissions.sat_scores.50th_percentile.math",
"latest.admissions.act_scores.50th_percentile.cumulative",
"latest.admissions.act_scores.50th_percentile.english",
"latest.admissions.act_scores.50th_percentile.math",
"latest.admissions.sat_scores.average.overall",
"latest.admissions.act_scores.midpoint.cumulative"
```
~58% is null and overlaps with ```latest.admissions.admission_rate.overall``` and ```latest.school.open_admissions_policy```



In [ ]:
# tdf = collect()
tdf = pd.read_csv(root/"data"/"clean"/"scorecard"/"clean_ml_scorecard_FL_programs.csv")
display(tdf.head())
display(tdf.info())

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\sebas\\PycharmProjects\\Git\\Seb_branch\\data\\clean\\scorecard\\clean_ml_scorecard_FL_programs.csv'

# Why so many missing Admission Rates?

In [5]:
df = tdf.copy()
round(df.isnull()["latest.admissions.admission_rate.overall"].mean(),4)

0.4226

In [6]:
round(df[df["latest.school.open_admissions_policy"]==2].isnull()["latest.admissions.admission_rate.overall"].mean(),4)

0.023

Approximately 40% of institutions have missing values for admission_rate.overall.
According to IPEDS reporting rules, institutions with an open admissions policy do not report traditional selectivity metrics such as admission rate or standardized test scores.

A check confirms that nearly all non-open-admission institutions report admission rates, indicating that the missingness is structural rather than random.

Therefore, missing admission rates are interpreted as corresponding primarily to open-admission institutions, and the open_admissions_policy variable is retained to preserve this structural distinction.

In [7]:
df = df.drop(columns="id")
df = clean(df)

Numeric columns: Index(['distance', 'credential_level', '4_yr_median_earnings',
       '4_yr_working_count', 'admission_rate_overall', 'median_family_income',
       'students_with_pell_grant'],
      dtype='object')


In [8]:
df["selectivity_bucket"] = pd.cut(
    df["admission_rate_overall"],
    bins=[0, 0.3, 0.7, 1],
    labels=["elite", "mid", "open"]
)

In [9]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2501 entries, 0 to 2500
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   code                       2501 non-null   string  
 1   title                      2501 non-null   string  
 2   unit_id                    2501 non-null   string  
 3   distance                   2501 non-null   int64   
 4   school_type                2501 non-null   string  
 5   credential_level           2501 non-null   int64   
 6   4_yr_median_earnings       2501 non-null   int64   
 7   4_yr_working_count         2501 non-null   int64   
 8   school_name                2501 non-null   string  
 9   locale                     2501 non-null   string  
 10  carnegie_size_setting      2501 non-null   string  
 11  admission_rate_overall     1444 non-null   Float64 
 12  median_family_income       2491 non-null   float64 
 13  students_with_pell_grant   2303 n

None

In [214]:
df.head()

,code,title,unit_id,distance,school_type,credential_level,4_yr_median_earnings,4_yr_working_count,school_name,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket
0,1205,Culinary Arts and Related Services.,132374,1,Public,1,22265,28,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1,NaN
1,4603,Electrical and Power Transmission Installers.,132374,2,Public,1,41177,27,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1,NaN
2,4702,"Heating, Air Conditioning, Ventilation and Ref...",132374,1,Public,1,47325,48,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1,NaN
3,4706,Vehicle Maintenance and Repair Technologies/Te...,132374,1,Public,1,42839,28,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1,NaN
4,5108,Allied Health and Medical Assisting Services.,132374,1,Public,1,33081,29,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1,NaN


In [10]:
save(clean(df),"clean","ml_scorecard_FL_programs.csv")
save(tdf,file_name="ml_scorecard_FL_programs.csv")

Numeric columns: Index(['distance', 'credential_level', '4_yr_median_earnings',
       '4_yr_working_count', 'admission_rate_overall', 'median_family_income',
       'students_with_pell_grant'],
      dtype='object')


In [11]:
display("Relevant numeric variables statistics:", df.select_dtypes(exclude=object).describe())
display("Missing values per column:", df.isna().sum())
display("Correlation matrix:", df.corr(numeric_only=True))

'Relevant numeric variables statistics:'

,distance,credential_level,4_yr_median_earnings,4_yr_working_count,admission_rate_overall,median_family_income,students_with_pell_grant
count,2501.000000,2501.000000,2501.000000,2501.000000,1444.0,2491.000000,2303.0
mean,1.431028,4.064774,62386.475010,132.365054,0.53513,29262.739864,0.734936
std,0.829784,10.392411,25274.413965,382.210450,0.228989,13060.807166,0.127915
min,0.000000,1.000000,10611.000000,16.000000,0.189,0.000000,0.192053
25%,1.000000,2.000000,46115.000000,26.000000,0.4011,21349.000000,0.667383
50%,1.000000,3.000000,57448.000000,46.000000,0.5466,25118.000000,0.721905
75%,2.000000,3.000000,75539.000000,111.000000,0.6966,38662.000000,0.82967
max,3.000000,99.000000,221571.000000,9437.000000,1.0,81806.000000,0.990401


'Missing values per column:'

code                            0
title                           0
unit_id                         0
distance                        0
school_type                     0
credential_level                0
4_yr_median_earnings            0
4_yr_working_count              0
school_name                     0
locale                          0
carnegie_size_setting           0
admission_rate_overall       1057
median_family_income           10
students_with_pell_grant      198
open_admissions_policy          4
age_entry                      10
title_iv_eligibility_type       0
selectivity_bucket           1057
dtype: int64

'Correlation matrix:'

,distance,credential_level,4_yr_median_earnings,4_yr_working_count,admission_rate_overall,median_family_income,students_with_pell_grant
distance,1.000000,-0.153944,0.092568,0.073129,0.132306,-0.054622,0.056554
credential_level,-0.153944,1.000000,0.048096,-0.029798,-0.217481,0.005684,0.049736
4_yr_median_earnings,0.092568,0.048096,1.000000,0.025095,-0.215881,0.333229,-0.311550
4_yr_working_count,0.073129,-0.029798,0.025095,1.000000,0.058224,-0.059164,0.050633
admission_rate_overall,0.132306,-0.217481,-0.215881,0.058224,1.000000,-0.416269,0.393917
median_family_income,-0.054622,0.005684,0.333229,-0.059164,-0.416269,1.000000,-0.905237
students_with_pell_grant,0.056554,0.049736,-0.311550,0.050633,0.393917,-0.905237,1.000000


In [12]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score

# Reproducibility
RANDOM_STATE = 42

In [13]:
model_df = df.copy()
target = '4_yr_median_earnings'


X = model_df.drop(columns=[target]).copy()
X = X.drop(columns=["title","4_yr_working_count"])
y = pd.to_numeric(model_df[target], errors='coerce').copy()

cat_cols = [
    'code',
    'school_type',
    'locale',
    'carnegie_size_setting',
    'open_admissions_policy',
    'title_iv_eligibility_type',
    'credential_level',
    'distance',
    "selectivity_bucket"
]
num_cols = [
    'admission_rate_overall',
    'median_family_income',
    'students_with_pell_grant',
    'age_entry'
]

# categorical: force plain object and replace missing with np.nan
for c in cat_cols:
    X[c] = X[c].astype(object)
    X[c] = X[c].replace({pd.NA: np.nan})

# numeric: force numeric with np.nan
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce')

# nuclear option: remove any lingering pd.NA anywhere in X
X = X.astype(object).replace({pd.NA: np.nan})

# now restore numeric cols back to numeric dtype
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce')

In [14]:
for c in X.columns:
    bad = X[c].map(lambda v: type(v).__name__).eq('NAType').sum()
    if bad:
        print(c, bad)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

In [ ]:
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols),
])

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', Ridge())
])

param_grid = {
    'clf__alpha': [0.01, 0.1, 1.0, 3.0, 10.0, 50.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

y_train_log=np.log(y_train)
y_test_log=np.log(y_test)

grid.fit(X_train, y_train_log)

print("Best params:", grid.best_params_)
print("Best CV MAE:", round(-grid.best_score_, 2))
best_model = grid.best_estimator_
preds = best_model.predict(X_test)

print("Test MAE:", round(mean_absolute_error(y_test_log, preds), 2))
print("Test R2:", round(r2_score(y_test_log, preds), 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'clf__alpha': 0.1}
Best CV MAE: 0.14
Test MAE: 0.14
Test R2: 0.7616


In [225]:
baseline_pred = [y_train_log.mean()] * len(y_test_log)

print("Baseline MAE:", round(mean_absolute_error(y_test_log, baseline_pred), 2))

Baseline MAE: 0.31


In [237]:
df["4_yr_median_earnings"].skew()

1.3098372365114241

In [238]:
np.log(df["4_yr_median_earnings"]).skew()

-0.19400898905945482

In [217]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict


pipe_rfr = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

param_grid_rfr = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [None, 5, 10, 20],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4]
}

grid_rfr = GridSearchCV(
    estimator=pipe_rfr,
    param_grid=param_grid_rfr,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

grid_rfr.fit(X_train, y_train_log)

print("RF Best params:", grid_rfr.best_params_)
print("RF Best CV MAE:", round(-grid_rfr.best_score_, 2))

best_rfr = grid_rfr.best_estimator_
rfr_preds = best_rfr.predict(X_test)

print("RF Test MAE:", round(mean_absolute_error(y_test_log, rfr_preds), 2))
print("RF Test R2:", round(r2_score(y_test_log, rfr_preds), 4))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
RF Best params: {'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2, 'clf__n_estimators': 200}
RF Best CV MAE: 0.15
RF Test MAE: 0.15
RF Test R2: 0.7356


In [218]:
from xgboost import XGBRegressor

pipe_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', XGBRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    ))
])

param_grid_xgb = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [None, 5, 10, 20],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__subsample': [0.8, 1.0]
}

grid_xgb_log = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

grid_xgb_log.fit(X_train, y_train_log)

print("XGB Best params:", grid_xgb_log.best_params_)
print("XGB Best CV MAE:", round(-grid_xgb_log.best_score_, 2))

best_xgb_log = grid_xgb_log.best_estimator_
xgb_preds_log = best_xgb_log.predict(X_test)

print("XGB Test MAE:", round(mean_absolute_error(y_test_log, xgb_preds_log), 2))
print("XGB Test R2:", round(r2_score(y_test_log, xgb_preds_log), 4))

Fitting 5 folds for each of 72 candidates, totalling 360 fits
XGB Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 10, 'clf__n_estimators': 300, 'clf__subsample': 0.8}
XGB Best CV MAE: 0.13
XGB Test MAE: 0.13
XGB Test R2: 0.7912


In [210]:
display(np.log(y_train.head()))

2123    10.144707
2280    11.274440
1085    10.810596
1868    10.900049
1412    10.944806
Name: 4_yr_median_earnings, dtype: float64

In [213]:
y_test_log=np.log(y_test)

best_xgb_log = grid_xgb_log.best_estimator_
xgb_preds_log = best_xgb_log.predict(X_test)

print("XGB Test MAE:", round(mean_absolute_error(y_test_log, xgb_preds_log), 2))
print("XGB Test R2:", round(r2_score(y_test_log, xgb_preds_log), 4))

XGB Test MAE: 0.13
XGB Test R2: 0.7912


All models significantly outperformed the baseline. The linear model and Random Forest performed similarly, suggesting that linear relationships explain a large portion of the variance. However, XGBoost achieved the best performance, reducing MAE from ~8,700 to ~7,900 and increasing R2 to 0.736. This indicates that nonlinear interactions exist in the data and are effectively captured by gradient boosting methods.

In [ ]:
feature_names = best_xgb_log.named_steps["preprocessor"].get_feature_names_out()

importances = best_xgb_log.named_steps["clf"].feature_importances_

importance = pd.Series(importances, index=feature_names)

print("most important features")
display(importance.sort_values(ascending=False).head(15))
print("least important features")
display(importance.sort_values(ascending=True).head(15))

most important features


cat__credential_level_1          0.111381
cat__open_admissions_policy_2    0.059600
cat__code_1204                   0.057876
cat__code_5138                   0.036228
cat__credential_level_7          0.022290
cat__credential_level_5          0.016697
cat__code_1101                   0.014256
cat__credential_level_6          0.014101
cat__code_5007                   0.013132
cat__code_5005                   0.012843
cat__code_5009                   0.012256
cat__code_1409                   0.012082
cat__code_1410                   0.011712
cat__code_5006                   0.011271
cat__code_4901                   0.011058
dtype: float32

least important features


cat__selectivity_bucket_open        0.0
cat__code_1203                      0.0
cat__code_0904                      0.0
cat__carnegie_size_setting_18       0.0
cat__code_4299                      0.0
cat__carnegie_size_setting_4        0.0
cat__code_4227                      0.0
cat__carnegie_size_setting_7        0.0
cat__code_4004                      0.0
cat__code_0305                      0.0
cat__code_0110                      0.0
cat__selectivity_bucket_elite       0.0
cat__title_iv_eligibility_type_5    0.0
cat__code_3001                      0.0
cat__code_0101                      0.0
dtype: float32

Feature importance analysis from the XGBoost model shows that categorical variables, particularly program codes (CIP), credential level, admission policy, and carnegie classification, were the most influential predictors. Additionally, several features had zero importance, indicating that certain categories did not contribute meaningfully to prediction, likely due to lack of signal.

In [ ]:
cv_preds = cross_val_predict(grid_xgb_log, X, np.log(y), cv=cv, method="predict")

log_error_df = X.copy()
log_error_df["actual"] = np.log(y)
log_error_df["pred"] = cv_preds
# log_error_df["error"] = log_error_df["actual"] - log_error_df["pred"]

cols_to_add = ["title", "school_name", "credential_level","4_yr_working_count"]
log_error_df[cols_to_add] = model_df.loc[log_error_df.index, cols_to_add]

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Fitting 5 folds for each of 72 candidates, totalling 360 fits


The model was trained on log-transformed earnings to address skewness and improve predictive stability. Predictions were then exponentiated back to the original scale to evaluate performance and interpret errors in dollar terms.

Predictions are the typical (median-like) expected earnings rather than the average, which is appropriate given the skewed nature of income data.

In [260]:
error_df=log_error_df.copy()
display(error_df.head())

delog_col = ["actual","pred"]
for col in delog_col:
    error_df[col] = np.exp(error_df[col])

error_df["error"] = error_df["actual"] - error_df["pred"]

display(error_df.head())

,code,unit_id,distance,school_type,credential_level,school_name,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,actual,pred,error,title,4_yr_working_count
0,1205,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.010771,10.388908,-0.378137,Culinary Arts and Related Services.,28
1,4603,132374,2,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.625635,10.623437,0.002198,Electrical and Power Transmission Installers.,27
2,4702,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.764794,10.637287,0.127507,"Heating, Air Conditioning, Ventilation and Ref...",48
3,4706,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.665204,10.842904,-0.177700,Vehicle Maintenance and Repair Technologies/Te...,28
4,5108,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.406714,10.442400,-0.035686,Allied Health and Medical Assisting Services.,29


,code,unit_id,distance,school_type,credential_level,school_name,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,actual,pred,error,title,4_yr_working_count
0,1205,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,22265.0,32497.171875,-10232.171875,Culinary Arts and Related Services.,28
1,4603,132374,2,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,41177.0,41086.582031,90.417969,Electrical and Power Transmission Installers.,27
2,4702,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,47325.0,41659.601562,5665.398438,"Heating, Air Conditioning, Ventilation and Ref...",48
3,4706,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,42839.0,51169.757812,-8330.757813,Vehicle Maintenance and Repair Technologies/Te...,28
4,5108,132374,1,Public,1,Atlantic Technical College,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,33081.0,34282.832031,-1201.832031,Allied Health and Medical Assisting Services.,29


In [ ]:
error_df=error_df[error_df["4_yr_working_count"] >= 20]

valid_codes = (
    error_df.groupby("code")["school_name"]
    .nunique()
)

valid_codes = valid_codes[valid_codes >= 4].index
error_df = error_df[error_df["code"].isin(valid_codes)]

error_df["school_count"] = error_df.groupby(["code","credential_level"])["school_name"].transform("nunique")

error_df["confidence"] = pd.cut(
    error_df["school_count"],
    bins=[0, 5, 15, 100],
    labels=["low", "medium", "high"]
)

error_df["pct_error"] = error_df["error"] / error_df["pred"]

k = np.percentile(np.log1p(error_df["4_yr_working_count"]), 75)

weight = np.log1p(error_df["4_yr_working_count"])/np.log1p(error_df["4_yr_working_count"]+k)

error_df["score"] = (
    error_df["pct_error"] * weight
)

In [320]:
np.percentile(np.log1p(error_df["4_yr_working_count"]), [25, 50, 75, 90, 95])

array([3.4657359 , 4.04305127, 4.87519732, 5.71373281, 6.14825234])

In [330]:
rank_pct = error_df["pct_error"].rank()
rank_score = error_df["score"].rank()

adjustments = (rank_pct - rank_score).abs().mean()
adjustments/len(error_df)

0.0

In [1]:
error_df["rank_diff"] = (rank_score - rank_pct)

# display(error_df.sort_values("score").head(10))
error_df.sort_values("rank_diff").tail(10)
error_df.sort_values("rank_diff").head(10)

NameError: name 'rank_score' is not defined

“The weighting scheme introduces only minor adjustments to ranking positions, suggesting that prediction error remains dominant while program size provides a secondary refinement.”

To avoid instability in groups with small sample sizes, a small constant (epsilon) was added to the standard deviation when computing the final score. This prevents artificially inflated scores caused by near-zero variance estimates, while still allowing all groups to be included in the analysis.

To account for differences in sample size, a soft penalization factor was applied using sqrt(n / (n + k)). This approach reduces the influence of groups with small sample sizes without excluding them entirely. As n increases, the penalty diminishes, allowing larger groups to retain their full weight while appropriately down-weighting less reliable estimates.

In [331]:
program_variability = (
    error_df
    .groupby(["code","credential_level"])
    .agg(
        std_score=("score", "std"),
        mean_score=("score", "mean"),
        min_score=("score", "min"),
        max_score=("score", "max"),
        n=("score", "size")
    )
)

epsilon=0.064

program_variability["relative_variability"] = (
    program_variability["std_score"]/(program_variability["mean_score"].abs()+epsilon)
)
program_variability["rv_weighted"] = (
    (program_variability["relative_variability"])
    * (program_variability["n"]/(program_variability["n"]+10)) 
)

top_programs=program_variability.sort_values("rv_weighted", ascending=False).head(10)
top_programs

,,std_score,mean_score,min_score,max_score,n,relative_variability,rv_weighted
code,credential_level,,,,,,,
5107,2,0.340952,0.003049,-0.250975,1.167195,15,5.085124,3.051074
5005,3,0.345861,0.003761,-0.495906,0.604122,11,5.104167,2.673611
5122,5,0.280151,-0.003616,-0.369115,0.673778,11,4.143239,2.170268
5135,1,0.196055,0.004639,-0.463078,0.436477,27,2.856330,2.084349
5009,3,0.269436,-0.006662,-0.531228,0.518205,10,3.813031,1.906515
1204,1,0.213741,0.036900,-0.462573,0.965444,81,2.118355,1.885569
5108,2,0.186953,-0.005916,-0.542007,0.411584,23,2.673965,1.863672
1310,5,0.336449,0.022326,-0.275778,0.807405,8,3.897432,1.732192
5202,2,0.166749,0.009344,-0.331713,0.256102,26,2.273509,1.641978


In [332]:
(
    error_df[(error_df["code"] == "5122")&(error_df["credential_level"]==5)]
    .sort_values(["credential_level", "score"], ascending=[True, False])
)

,code,unit_id,distance,school_type,credential_level,school_name,locale,carnegie_size_setting,admission_rate_overall,median_family_income,...,actual,pred,error,title,4_yr_working_count,school_count,confidence,pct_error,score,rank_diff
1946,5122,138354,3,Public,5,University of West Florida,41,12,0.5824,34583.0,...,111363.0,66533.898438,44829.101562,Public Health.,32,11,medium,0.673778,0.673778,0.0
1336,5122,136215,2,"Private, nonprofit",5,Nova Southeastern University,21,16,0.7321,31002.0,...,98338.0,83508.648438,14829.351562,Public Health.,91,11,medium,0.177579,0.177579,6.0
2059,5122,408844,3,"Private, for-profit",5,Florida National University-Main Campus,21,9,NaN,13365.0,...,64291.0,58523.089844,5767.910156,Public Health.,22,11,medium,0.098558,0.098558,-16.0
1715,5122,137351,1,Public,5,University of South Florida,11,15,0.4319,32495.0,...,79273.0,72645.726562,6627.273438,Public Health.,242,11,medium,0.091227,0.091227,4.0
569,5122,133951,3,Public,5,Florida International University,21,15,0.5466,23924.0,...,73457.0,74947.171875,-1490.171875,Public Health.,70,11,medium,-0.019883,-0.019883,-1.0
869,5122,134130,3,Public,5,University of Florida,12,16,0.2420,39127.0,...,80455.0,84623.984375,-4168.984375,Public Health.,109,11,medium,-0.049265,-0.049265,-1.0
722,5122,134097,1,Public,5,Florida State University,12,15,0.2422,43729.0,...,71264.0,77165.070312,-5901.070313,Public Health.,44,11,medium,-0.076473,-0.076473,4.0
1277,5122,136172,1,Public,5,University of North Florida,11,15,0.5324,37074.0,...,63367.0,71498.046875,-8131.046875,Public Health.,25,11,medium,-0.113724,-0.113724,12.0
1481,5122,136950,1,"Private, nonprofit",5,Rollins College,21,11,0.4754,43978.0,...,76051.0,86736.929688,-10685.929688,Public Health.,32,11,medium,-0.123199,-0.123199,6.0
312,5122,133650,2,Public,5,Florida Agricultural and Mechanical University,12,13,0.2056,28781.0,...,47272.0,70477.703125,-23205.703125,Public Health.,26,11,medium,-0.329263,-0.329263,3.0


In [305]:
import plotly.express as px
bar_df = error_df[
    (error_df["code"] == "5122") & (error_df["credential_level"] == 5)
]

bar_df = bar_df.sort_values(by="error", ascending=True)

px.bar(
    bar_df,
    x="error",
    y="school_name",
    orientation="h",
    labels={"error":"Schools","school_name":"Pred. Error"},
    title="Master's in Public Health"
)

Programs such as Public Health show substantial variability in outcomes across institutions, even after controlling for observable factors. This suggests that institutional effects such as program quality, networking opportunities, or industry connections may play a significant role in shaping student outcomes.

In [ ]:
error_df.sort_values